# Filesystems

> For filesystems that store video, text, and more

In [ ]:
# | default_exp filesystems


In [ ]:
# | hide
from nbdev.showdoc import *

In [ ]:
# | export
from typing import Generator, Optional, BinaryIO, IO, Any
from pydantic import BaseModel
from product_horse.db import User
from dataclasses import dataclass, field
from enum import Enum
from abc import ABC, abstractmethod
import shutil
from contextlib import contextmanager

import io
from storage_client import StorageClient

from uuid import uuid4

In [ ]:
# | export
class File(BaseModel):
    uri: str
    content: Optional[bytes | io.BytesIO] = None

    class Config:
        arbitrary_types_allowed = True


class AudioFile(File):
    start: int
    end: int
    file: str


class VideoFile(File):
    start: int
    end: int
    file: str


class TextFile(File):
    pass

In [ ]:
# | export
class FileSystemType(Enum):
    LOCAL = "local"
    R2 = "r2"


class FilePathType(Enum):
    """Files can come in as Audio, Video, Text, or PDF.
    The original file should be saved in user_id/originals.
    Transformed files should be saved in user_id/audio, user_id/video, user_id/text, user_id/pdf
    Relationships back to original file should be saved in the db, but that's not handled by filesystem APIs"""

    USER_ID_BASE = "user_id"
    USER_ID_BASE_AUDIO = "user_id_audio"
    USER_ID_BASE_VIDEO = "user_id_video"
    USER_ID_BASE_CLIPS = "user_id_clips"
    USER_ID_BASE_TEXT = "user_id_text"
    USER_ID_BASE_PDF = "user_id_pdf"


@dataclass
class AbstractFileSystem(ABC):
    base_path: str = field(default="")

    @property
    @abstractmethod
    def tenant_video_path(self) -> str:
        """The path for tenant-scoped videos"""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def build_user_path(self, user: User, file_path_type: FilePathType) -> str:
        """Build the file path according to the inputted objects and rules.
        Does not create resources, just a string formatter"""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def create_folder(self, path: str) -> str:
        """Creates folders at the specified location. Doesn't raise errors if locaqtion exists"""
        raise NotImplementedError("This method should be implemented by subclasses")

    def sanitize_characters(self, text: str) -> str:
        """Replace non-ASCII characters with their Unicode codepoint representations."""
        return (
            text.encode("unicode_escape")
            .decode()
            .replace("\\u", "u")
            .replace("\\", "_")
            .replace("\\U", "U")
        )

    def sanitize_filename(self, filename: str) -> str:
        """Use as an add-on to sanitize characters. Replace non-ASCII characters with their Unicode codepoint representations."""
        return filename.replace("/", "_").replace("\\", "_").replace(" ", "_")

    @abstractmethod
    def validate_access(self, user: User, path: str) -> bool:
        """Check if the user has access to the specified location"""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def check_exists(self, path: str) -> bool:
        """Check if the file exists at the specified path."""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    async def create_file(
        self,
        path: str,
        content: bytes | io.BytesIO,
        name: str,
        authorized: bool = False,
    ) -> File:
        """Creates files at specified path. Throws an error if the location doesn't exist.
        Does NOT check permissions — pass in authorized to allow access"""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def read_file(self, path: str, load_content: bool = False) -> File:
        """Reads the file at the specified path.
        If load_content is True, the content of the file is loaded into memory.
        Otherwise, the content is set to None."""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    async def update_file(self, path: str, content: bytes) -> File:
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def delete_file(self, path: str) -> File:
        raise NotImplementedError("This method should be implemented by subclasses")
    
    @abstractmethod
    def file_stream(self, path: str, mode: str = "rb") -> BinaryIO | IO[Any]:
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    @contextmanager
    def temporary_user_directory(self, user: User) -> Generator[str, None, None]:
        """Expose a temporary directory for the user to store files in.
        This directory should be deleted after the user has finished with it.
        should be used as `with storage.temporary_user_directory(user) as tmp_dir:`"""
        raise NotImplementedError("This method should be implemented by subclasses")

In [ ]:
# | export
import os
import tempfile


@dataclass
class LocalFileSystem(AbstractFileSystem):
    base_path: str = field(
        default="",
        metadata={
            "help": "The base path for the local file system. Set to tenant_id for tenant-scoped storage"
        },
    )

    @property
    def tenant_video_path(self) -> str:
        return f"{self.base_path}/videos"

    def build_user_path(
        self: AbstractFileSystem, user: User, file_path_type: FilePathType
    ) -> str:
        if file_path_type == FilePathType.USER_ID_BASE:
            return f"{self.base_path}/{user.id}/originals"
        elif file_path_type == FilePathType.USER_ID_BASE_AUDIO:
            return f"{self.base_path}/{user.id}/audio"
        elif file_path_type == FilePathType.USER_ID_BASE_VIDEO:
            return f"{self.base_path}/{user.id}/video"
        elif file_path_type == FilePathType.USER_ID_BASE_TEXT:
            return f"{self.base_path}/{user.id}/text"
        elif file_path_type == FilePathType.USER_ID_BASE_PDF:
            return f"{self.base_path}/{user.id}/pdf"
        elif file_path_type == FilePathType.USER_ID_BASE_CLIPS:
            return f"{self.base_path}/{user.id}/clips"
        else:
            raise ValueError(f"Invalid file path type: {file_path_type}")

    def create_folder(self: AbstractFileSystem, path: str) -> str:
        path = self.sanitize_characters(path)
        os.makedirs(path, exist_ok=True)
        return path

    def validate_access(self: AbstractFileSystem, user: User, path: str) -> bool:
        # Simplified access validation: checks if path starts with user's directory
        return str(user.id) in path

    def check_exists(self: AbstractFileSystem, path: str) -> bool:
        return os.path.exists(path)

    async def create_file(
        self: AbstractFileSystem,
        path: str,
        content: bytes | io.BytesIO,
        name: str,
        authorized: bool = False,
    ) -> File:
        if not authorized:
            raise FileNotFoundError("Access denied")
        if not self.check_exists(os.path.dirname(path)):
            raise FileNotFoundError("Location doesn't exist")
        sanitized_name = self.sanitize_characters(self.sanitize_filename(name))
        if self.check_exists(path + "/" + sanitized_name):
            raise FileExistsError("File already exists — use update_file instead")
        if len(sanitized_name) == 0:
            raise ValueError("File name cannot be empty")
        with open(path + "/" + sanitized_name, "wb") as f:
            if isinstance(content, bytes):
                f.write(content)
            else:
                shutil.copyfileobj(content, f)
        return File(uri=path + "/" + sanitized_name)

    def read_file(
        self: AbstractFileSystem, path: str, load_content: bool = False
    ) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        if load_content:
            with open(path, "rb") as f:
                content = f.read()
        else:
            content = None
        return File(uri=path, content=content)
    
    def file_stream(self, path: str, mode: str = "rb") -> BinaryIO | IO[Any]:
        if not os.path.isfile(path) and mode != "wb":
            raise FileNotFoundError("File does not exist")
        return open(path, mode)

    async def update_file(self: AbstractFileSystem, path: str, content: bytes) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        return File(uri=path)

    def delete_file(self: AbstractFileSystem, path: str) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        os.remove(path)
        return File(uri=path)

    @contextmanager
    def temporary_user_directory(self, user: User) -> Generator[str, None, None]:
        temp_dir = tempfile.mkdtemp()
        user_temp_dir = os.path.join(temp_dir, str(user.id))
        os.makedirs(user_temp_dir, exist_ok=True)
        try:
            yield user_temp_dir
        finally:
            shutil.rmtree(temp_dir)

In [ ]:
user = User(
    company_id=uuid4(), created_by_id=uuid4()
)  # Assuming you have a User object

file_system = LocalFileSystem()

with file_system.temporary_user_directory(user) as tmp_dir:
    # You can now use tmp_dir to store temporary files
    print(f"Temporary directory for user: {tmp_dir}")
    # Perform file operations within tmp_dir
    # Example: create a temporary file
    await file_system.create_file(
        tmp_dir, b"Hello, world!", "temp_file.txt", authorized=True
    )
    # Read the temporary file
    print(file_system.read_file(tmp_dir + "/temp_file.txt", load_content=True))
# The temporary directory and its contents are automatically cleaned up here

Temporary directory for user: /var/folders/qj/d65y3xj97dj5gmv5gpb9bq3m0000gn/T/tmpi7l3zjrz/cfd7ef4c-61c8-48b8-bda4-38137ba90cd9
uri='/var/folders/qj/d65y3xj97dj5gmv5gpb9bq3m0000gn/T/tmpi7l3zjrz/cfd7ef4c-61c8-48b8-bda4-38137ba90cd9/temp_file.txt' content=b'Hello, world!'


In [ ]:
# | export

MAX_PART_SIZE = 10 * 1024 * 1024


@dataclass
class R2StorageClient(AbstractFileSystem):
    client: StorageClient = field(init=False)
    base_path: str = field(
        default="",
        metadata={
            "help": "The base path for the local file system. Set to company_id for tenant-scoped storage"
        },
    )
    jwt: str = field(
        default="",
        metadata={"help": "The JWT for the storage client"},
    )
    api_url: str = field(
        default="https://storage.producthorse.workers.dev/",
        metadata={"help": "The API URL for the storage client"},
    )

    def __post_init__(self):
        self.client = StorageClient(base_url=self.api_url, jwt=self.jwt)

    @property
    def tenant_video_path(self) -> str:
        return f"{self.base_path}/videos"

    def build_user_path(self, user: User, file_path_type: FilePathType) -> str:
        return f"{self.base_path}/{user.id}/{file_path_type.value}"

    def create_folder(self, path: str) -> str:
        # R2 doesn't need folder creation
        return path

    def validate_access(self, user: User, path: str) -> bool:
        # Access validation should be handled by the R2 worker
        return True

    def check_exists(self, path: str) -> bool:
        try:
            self.client.get_file_metadata(path)
            return True
        except Exception:
            return False

    async def create_file(
        self,
        path: str,
        content: bytes | io.BytesIO,
        name: str,
        authorized: bool = False,
        visibility: str = "internal",
        mime_type: str = "text/plain",
    ) -> File:
        if not authorized:
            raise FileNotFoundError("Access denied")
        if isinstance(content, bytes):
            content = io.BytesIO(content)
        path_to_write = path + "/" + name
        response = await self.client.upload_file(
            content, path_to_write, visibility, mime_type
        )
        if response is not None:
            return File(uri=path_to_write)
        else:
            raise Exception("Failed to create file")

    def read_file(self, path: str, load_content: bool = False) -> File:
        response = self.client.download_file(path)
        content = response if load_content else None
        if content is None:
            return File(uri=path)
        return File(uri=path, content=content.read())

    async def update_file(self, path: str, content: bytes) -> File:
        # For R2, updating is the same as creating
        return await self.create_file(
            path, content, os.path.basename(path), authorized=True
        )

    def delete_file(self, path: str) -> File:
        raise NotImplementedError("This client doesn't support file deletion yet.")
    
    def get_signed_url(self, path: str) -> str:
        return self.client.get_signed_url(path)
    
    def file_stream(self, path: str, mode: str = "rb") -> BinaryIO | IO[Any]:
        if mode != "rb":
            raise NotImplementedError("R2StorageClient only supports read mode ('rb')")

        return self.client.download_file(path)
    
    @contextmanager
    def temporary_user_directory(self, user: User) -> Generator[str, None, None]:
        # R2 doesn't have directories, raise an error
        raise NotImplementedError("This client doesn't support temporary directories.")

        # temp_path = f"temp/{user.id}/{uuid4()}"
        # try:
        #     yield temp_path
        # finally:
        #     pass
            # Clean up any files created in this temporary path - not implemented yet
            # files = self.client.get_files(prefix=temp_path)
            # for file in files:
            #     self.delete_file(file.key)

Below is the canonical FileSystem test. If a state machine passes the test below, it functions as intended.

Use it as a reference for how to use the filesystem apis and the order in which to use it.

Right now it's broken -- takes a really long time to work.

In [ ]:
# from hypothesis import strategies as st
# from hypothesis.stateful import (
#     RuleBasedStateMachine,
#     rule,
#     precondition,
#     Bundle,
#     consumes,
# )
# from product_horse.core import run_test_case_with_pdb


# @st.composite
# def unique_filename_strategy(draw):
#     base_name = draw(
#         st.text(min_size=1, alphabet=st.characters(blacklist_characters="/\\"))
#     )
#     unique_suffix = draw(st.integers(min_value=0))
#     return f"{base_name}_{unique_suffix}"


# class FileSystemStateMachine(RuleBasedStateMachine):
#     def __init__(self):
#         super().__init__()
#         self.file_system = LocalFileSystem(base_path=tempfile.mkdtemp())
#         self.file_count = 0
#         self.user_count = 0

#     def teardown(self):
#         assert "tmp" in self.file_system.base_path
#         try:
#             shutil.rmtree(self.file_system.base_path)
#         except FileNotFoundError:
#             pass

#     files = Bundle("files")
#     user_paths = Bundle("user_paths")
#     file_paths = Bundle("file_paths")

#     @rule(
#         target=user_paths,
#         user=st.builds(User),
#         file_path_type=st.sampled_from(FilePathType),
#     )
#     def build_user_path(self, user: User, file_path_type: FilePathType) -> str:
#         """Format the correct user path for the file system and file type, check access"""
#         self.user_count += 1
#         path = self.file_system.build_user_path(user, file_path_type)
#         assert self.file_system.base_path in path
#         assert self.file_system.validate_access(user, path)
#         return path

#     @rule(target=file_paths, path=user_paths)
#     @precondition(lambda self: self.user_count > 0)
#     def create_folder(self, path: str) -> str:
#         """Create a folder in the file system for the user, based on user_path.
#         Returns no exception if path exists already."""
#         created_path = self.file_system.create_folder(path)
#         assert os.path.exists(created_path)
#         return created_path

#     @rule(
#         target=files,
#         path=file_paths,
#         content=st.binary(min_size=1),
#         filename=unique_filename_strategy(),
#     )
#     def create_file(self, path: str, content: bytes, filename: str) -> File:
#         """Create a file in the file system."""
#         file = self.file_system.create_file(
#             path, content, name=filename, authorized=True
#         )
#         assert self.file_system.check_exists(file.uri)
#         self.file_count += 1
#         return file

#     @rule(file=consumes(files))
#     @precondition(lambda self: self.file_count > 0)
#     def delete_file(self, file: File):
#         self.file_system.delete_file(file.uri)
#         self.file_count -= 1

#     @rule(file=files)
#     @precondition(lambda self: self.file_count > 0)
#     def read_file(self, file: File):
#         assert self.file_system.read_file(file.uri).uri == file.uri

#     @rule(file=files, content=st.binary(min_size=1))
#     @precondition(lambda self: self.file_count > 0)
#     def update_file(self, file: File, content: bytes):
#         self.file_system.update_file(file.uri, content)
#         assert self.file_system.check_exists(file.uri)


# TestFileSystem = FileSystemStateMachine.TestCase
# run_test_case_with_pdb(TestFileSystem, pdb_flag=False)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore